# Claude Agent SDK — playground v2

Rebuilt to be robust to two things that broke v1:

1. **Stale module.** Cell 1 purges `dspy_components.*` from `sys.modules` before
   importing, so you always get what is on disk — even in a long-lived kernel.
2. **Cross-cell dependencies.** Every lesson re-derives what it needs. You can run
   any lesson on its own after cell 1, in any order.

---

**Cells 1-2 and lessons 6-7 are free.** Lessons 2-5 cost a few cents each.
Lesson 8 runs a real extraction (~$0.55) and is behind a flag.

**Kernel:** `/home/ubuntu/miniconda3/envs/topics/bin/python`


## 1. Fresh import (run this first, and again after any backend edit)


In [ ]:
import os, sys, json, asyncio, tempfile, textwrap, dataclasses, importlib
from pathlib import Path

REPO, BACKEND = Path('/home/ubuntu/evistream'), Path('/home/ubuntu/evistream/backend')
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

os.environ.setdefault('AWS_SECRETS_NAME', 'evistream/production')
os.environ.setdefault('AWS_REGION', 'us-east-1')

# Hard purge: reload() alone can leave stale submodules behind.
for _m in [k for k in list(sys.modules) if k.startswith('dspy_components')]:
    del sys.modules[_m]

from utils.secrets_loader import load_secrets
load_secrets()
from dspy_components import agentic_table as A

import claude_agent_sdk as sdk
from claude_agent_sdk import (
    query, ClaudeAgentOptions, AssistantMessage, ResultMessage,
    TextBlock, ToolUseBlock, ToolResultBlock, tool, create_sdk_mcp_server,
    ToolAnnotations, HookMatcher,
)

MODEL = 'claude-sonnet-5'      # bare id: the SDK does NOT want the 'anthropic/' prefix

# Fail loudly and specifically if the module is older than this notebook expects.
_need = {'budget_usd_per_cell', 'budget_usd_session_cap',
         'budget_usd_extraction_cap', 'effort_col_switch'}
_have = {f.name for f in dataclasses.fields(A.Config())}
_missing = _need - _have
assert not _missing, (
    f'agentic_table is stale — missing {sorted(_missing)}. '
    f'Restart the kernel and re-run this cell.'
)

print('sdk         ', getattr(sdk, '__version__', '?'))
print('module      ', A.__file__)
print('api key set ', bool(os.environ.get('ANTHROPIC_API_KEY')))
print('caps        session ${:.2f} | extraction ${:.2f}'.format(
    A.CFG.budget_usd_session_cap, A.CFG.budget_usd_extraction_cap))
print('OK — module is current')


## 2. Spend guard + helpers


In [ ]:
NOTEBOOK_SPEND_CAP = 2.00     # mine, not Anthropic's. Checked BETWEEN calls only.
SPENT = 0.0

def _charge(x): 
    global SPENT
    SPENT += x or 0.0

def _check_budget():
    if SPENT >= NOTEBOOK_SPEND_CAP:
        raise RuntimeError(f'notebook cap reached: ${SPENT:.4f}')

async def run(prompt, options=None, show='full'):
    """Run one session. show='full' prints every block incl. tool inputs/results."""
    _check_budget()
    msgs, result = [], None
    async for m in query(prompt=prompt, options=options):
        msgs.append(m)
        for b in (getattr(m, 'content', None) or []):
            if isinstance(b, TextBlock) and b.text.strip() and show != 'quiet':
                print('  SAY   ', b.text.strip()[:400])
            elif isinstance(b, ToolUseBlock) and show != 'quiet':
                print('  CALL  ', b.name, json.dumps(b.input)[:220])
            elif isinstance(b, ToolResultBlock) and show == 'full':
                body = str(getattr(b, 'content', ''))[:220].replace(chr(10), ' ')
                print('  RESULT', f'is_error={getattr(b, "is_error", None)}', body)
        if isinstance(m, ResultMessage):
            result = m
    if result is not None:
        _charge(getattr(result, 'total_cost_usd', 0.0) or 0.0)
        print(f"  == {result.subtype} | turns={result.num_turns} "
              f"| ${getattr(result,'total_cost_usd',0) or 0:.4f} | total ${SPENT:.4f}")
    return msgs, result

def scratch(**files):
    d = Path(tempfile.mkdtemp(prefix='sdkplay_'))
    for k, v in files.items():
        (d / k.replace('__', '.')).write_text(v, encoding='utf-8')
    return d

PAPER_MD = textwrap.dedent('''
    # A trial of two mouthwashes
    ## Methods
    Patients were randomised to chlorhexidine or saline.
    ## Results
    | Arm            | n  | Plaque index at 6 weeks |
    |----------------|----|--------------------------|
    | Chlorhexidine  | 42 | 1.21 (0.30)              |
    | Saline         | 40 | 1.88 (0.41)              |

    Bleeding on probing was not measured in this study.
''')

def demo_opts(**kw):
    """Standard locked-down options over a fresh one-file scratch dir."""
    d = scratch(paper__md=PAPER_MD)
    base = dict(model=MODEL, cwd=str(d), tools=['Read', 'Grep'],
                allowed_tools=['Read', 'Grep'],
                disallowed_tools=['Write', 'Edit', 'Bash', 'WebSearch', 'WebFetch'],
                permission_mode='dontAsk', setting_sources=[],
                max_turns=8, max_budget_usd=0.25)
    base.update(kw)
    return ClaudeAgentOptions(**base), d


# ── Show a full extracted table ───────────────────────────────────────────────
# Printing only the anchor columns hides most of the result: an 11-column table
# is 5 identity columns and 6 measurements, and the measurements are the answer.
def show_table(res, field_def, max_quote=60):
    import pandas as pd
    fname = field_def['name']
    env = (res.envelope or {}).get(fname)
    rows = env.get('value') if isinstance(env, dict) else None
    if not isinstance(rows, list) or not rows:
        print('no rows —', json.dumps(env)[:300]); return None
    cols    = [c['field_name'] for c in field_def['subform_fields']]
    anchors = set(field_def.get('anchor_columns') or [])

    df = pd.DataFrame([{c: (r.get(c) or {}).get('value') for c in cols} for r in rows])
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 250)
    pd.set_option('display.max_colwidth', 28)
    print(f'{len(rows)} rows x {len(cols)} cols '
          f'({len(anchors)} anchor, {len(cols)-len(anchors)} value)')
    display(df)

    nr = df.isin(['NR']).sum()
    if nr.sum():
        print(f'NR cells: {int(nr.sum())} of {df.size} '
              f'({100*nr.sum()/df.size:.0f}%) — by column:')
        print(nr[nr > 0].sort_values(ascending=False).to_string())

    pages = [ (r.get(c) or {}).get('source_location', {}).get('page')
              for r in rows for c in cols ]
    got = [p for p in pages if p is not None]
    print(f'\ngrounded to a page: {len(got)} of {len(pages)} cells')
    return df


def show_cell(res, field_def, row=0, col=None):
    """Full detail for one cell: value, verbatim quote, resolved location."""
    fname = field_def['name']
    rows = (res.envelope or {}).get(fname, {}).get('value') or []
    if not rows: print('no rows'); return
    col = col or [c['field_name'] for c in field_def['subform_fields']][-1]
    print(json.dumps(rows[row].get(col), indent=2)[:900])
print('helpers ready | notebook cap ${:.2f}'.format(NOTEBOOK_SPEND_CAP))


## 3. Lesson — the smallest session  *(~$0.02)*

No tools. Shows the three message types and where cost lives.


In [ ]:
opts = ClaudeAgentOptions(model=MODEL, tools=[], max_turns=2, max_budget_usd=0.10,
                          system_prompt='Answer in one short sentence.')
msgs, res = await run('What is a systematic review?', opts)
print()
print('types     :', [type(m).__name__ for m in msgs])
print('session_id:', res.session_id)
print()
print('Note the ~2c floor: every session carries the harness preamble before it')
print('does any work. That is the fixed cost of an agent vs a plain API call.')


## 4. Lesson — tools, and seeing every call *(~$0.03)*

`show='full'` now prints tool **results** too, so you can tell a denied call from
a successful one — which v1 could not.


In [ ]:
opts, d = demo_opts()
_ = await run('Read paper.md. How many patients per arm? '
              'Was bleeding on probing measured? Grep before answering.', opts)


## 5. Lesson — structured output *(~$0.03)*

Constrains the final answer to a schema. This is what makes the extractor's
`{value, source_text, status}` envelope reliable instead of hoped-for.


In [ ]:
SCHEMA = {'type': 'object', 'additionalProperties': False,
          'required': ['arms'],
          'properties': {'arms': {'type': 'array', 'items': {
              'type': 'object', 'additionalProperties': False,
              'required': ['arm_label', 'n', 'source_text'],
              'properties': {'arm_label': {'type': 'string'},
                             'n': {'type': ['string', 'number']},
                             'source_text': {'type': 'string'}}}}}}

opts, d = demo_opts(output_format={'type': 'json_schema', 'schema': SCHEMA})
_, res = await run('Every treatment arm with its n. source_text verbatim.', opts, show='summary')
print()
print(json.dumps(getattr(res, 'structured_output', None), indent=2))


## 6. Lesson — your own tool, and forcing an order *(~$0.04)*

The tool runs in **this kernel**. The hook denies it out of order — this is
`commit_row_plan` + the plan gate, in miniature.


In [ ]:
CALLS, DENIED = [], []

@tool('register_rows', 'Register every row you found BEFORE extracting values.',
      {'type': 'object', 'required': ['rows'],
       'properties': {'rows': {'type': 'array', 'items': {'type': 'string'}}}},
      annotations=ToolAnnotations(readOnlyHint=True))
async def register_rows(args):
    CALLS.append(args['rows'])
    print('    >>> handler ran here:', args['rows'])
    return {'content': [{'type': 'text', 'text': f"Registered {len(args['rows'])} rows."}]}

@tool('emit_values', 'Emit the measured values. Requires register_rows first.',
      {'type': 'object', 'required': ['values'],
       'properties': {'values': {'type': 'string'}}},
      annotations=ToolAnnotations(readOnlyHint=True))
async def emit_values(args):
    return {'content': [{'type': 'text', 'text': 'Accepted.'}]}

srv = create_sdk_mcp_server(name='play', version='1.0.0',
                            tools=[register_rows, emit_values])

async def gate(input_data, tool_use_id, context):
    if (input_data or {}).get('tool_name') == 'mcp__play__emit_values' and not CALLS:
        DENIED.append('emit_values')
        print('    [hook] DENIED emit_values — no row plan yet')
        return {'hookSpecificOutput': {
            'hookEventName': 'PreToolUse', 'permissionDecision': 'deny',
            'permissionDecisionReason': 'Call register_rows first.'}}
    return {}

opts, d = demo_opts(
    mcp_servers={'play': srv},
    allowed_tools=['Read', 'Grep', 'mcp__play__register_rows', 'mcp__play__emit_values'],
    hooks={'PreToolUse': [HookMatcher(hooks=[gate])]},
    max_turns=10,
)
_ = await run('Read paper.md, then call emit_values with the plaque indices. '
              'Follow the tool descriptions.', opts)
print()
print('rows registered:', CALLS, '| denied:', DENIED)


## 7. Lesson — the real extractor's config  *(free)*

Self-contained: loads the schema itself. Shows how budget and effort are now derived.


In [ ]:
SCHEMA_PATH = REPO / 'eval/studies/ablation/base_schemas/dynamic_6c8eecce_ContinuousOutcomesV2.json'
schema_def = json.loads(SCHEMA_PATH.read_text())
sig_def, field_def = A.pick_table_field(schema_def)

anchors = set(field_def.get('anchor_columns') or [])
cols    = [c['field_name'] for c in field_def['subform_fields']]
n_val   = len([c for c in cols if c not in anchors])
print(f"field {field_def['name']}: {len(cols)} cols, {len(anchors)} anchors, {n_val} value cols")
print()

cfg = A.Config()
print('budget by shape (measured reference: 9col/6row = $0.539):')
for rows in (2, 6, 10, 20):
    cells = rows * n_val
    cap = min(cfg.budget_usd_session_cap, cfg.budget_usd_base + cells * cfg.budget_usd_per_cell)
    print(f'  {rows:>2} rows x {n_val} val cols = {cells:>3} cells -> ${cap:.3f}')
print()
eff = cfg.effort_large if n_val > cfg.effort_col_switch else cfg.effort_small
print(f'effort: {eff}  (keyed on value columns, which are known — not on a row guess)')
print(f'extraction cap ${cfg.budget_usd_extraction_cap:.2f} covers ALL sessions incl. repairs')


## 8. Lesson — run it for real  *(~$0.55, opt-in)*

Self-contained. Prints the cap **before** running.


In [ ]:
REALLY_RUN = False      # <- flip to True

SCHEMA_PATH = REPO / 'eval/studies/ablation/base_schemas/dynamic_6c8eecce_ContinuousOutcomesV2.json'
MD_PATH     = REPO / 'eval/sheets/markdown_perio/Artese 2015.md'

schema_def = json.loads(SCHEMA_PATH.read_text())
sig_def, field_def = A.pick_table_field(schema_def)
RUN_CFG = dataclasses.replace(A.CFG, trace=True)   # defaults are already sane now

anchors = set(field_def.get('anchor_columns') or [])
n_val   = len([c for c in field_def['subform_fields'] if c['field_name'] not in anchors])
cells   = 10 * n_val
print('will cap at ${:.3f} per session, ${:.2f} for the whole extraction'.format(
    min(RUN_CFG.budget_usd_session_cap, RUN_CFG.budget_usd_base + cells * RUN_CFG.budget_usd_per_cell),
    RUN_CFG.budget_usd_extraction_cap))

if not REALLY_RUN:
    print('skipped (REALLY_RUN is False)')
else:
    _check_budget()
    res = await A.extract_table_from_markdown(
        markdown_content=MD_PATH.read_text(), sig_def=sig_def,
        field_def=field_def, paper_hint=MD_PATH.stem, cfg=RUN_CFG)
    _charge(res.cost_usd)
    print()
    print(f'status     {res.status}')
    print(f'rows       emitted={res.n_rows} planned={res.plan_rows} '
          f'missing={res.rows_missing} withdrawn={res.rows_withdrawn}')
    print(f'repairs    {res.repair_rounds}  turns {res.num_turns}')
    print(f'quotes     checked={res.quotes_checked} failed={res.cells_failed_grounding}')
    print(f'cost       ${res.cost_usd:.4f} ({res.duration_ms/1000:.0f}s)')
    print(f'trace      {res.trace_path or "-"}')
    show_table(res, field_def)


## Open questions for the project

- Does `register_rows` (lesson 6) change what the model finds, or would it have
  found the same rows anyway? That is `commit_row_plan`'s entire justification and
  neither of us has measured it.
- Is `Grep` load-bearing? Re-run lesson 4 without it and see whether the
  "was bleeding measured?" answer stays honest.
- What *should* happen when a budget trips mid-extraction — return the rows
  collected so far, or nothing? Currently: nothing, reported as a visible failure.


---

## 9. Real run — Polat 2005b / CD015432 Continuous Outcomes v2  *(Ibuprofen)*

Uses the **live** `schema_def` straight from Supabase, not the snapshot in
`base_schemas/` — that snapshot is stale (it has no `anchor_columns`, the live
form does). This is the exact schema production would run.

| | |
|---|---|
| paper | `eval/sheets/markdown_Ibuprofen/Polat 2005b.md` (~9.8k tokens) |
| form | `CD015432 — Continuous Outcomes v2`, project **Ibuprofen** |
| field | `outcomes` — 11 cols, 5 anchors, **6 value columns** |
| anchors | comparison, outcome_type, reporter, scale, timepoint |

The form itself is still set to `standard`, which doesn't matter here — this cell
calls the extractor directly rather than going through the mode switch.

`trace=True` writes every tool call, quote check and repair round to
`/tmp/evistream_agentic_traces/`. Cell 10 renders it.


In [ ]:
REALLY_RUN = False        # <- flip to True

FORM_ID = 'afcee451-a73e-478d-8a83-98edb5cc4557'   # Ibuprofen (not the Testing copy)
MD_PATH = REPO / 'eval/sheets/markdown_Ibuprofen/Polat 2005b.md'

# Live schema_def — matches what production would load from the schemas table.
from utils.supabase_client import get_supabase_client
_row = (get_supabase_client().client.table('forms')
        .select('form_name,schema_name,schema_def')
        .eq('id', FORM_ID).execute().data[0])
schema_def = _row['schema_def']
if isinstance(schema_def, str):
    schema_def = json.loads(schema_def)
sig_def, field_def = A.pick_table_field(schema_def)

anchors = set(field_def.get('anchor_columns') or [])
cols    = [c['field_name'] for c in field_def['subform_fields']]
n_val   = len([c for c in cols if c not in anchors])

RUN_CFG = dataclasses.replace(A.CFG, trace=True)
cells   = 10 * n_val
sess_cap = min(RUN_CFG.budget_usd_session_cap,
               RUN_CFG.budget_usd_base + cells * RUN_CFG.budget_usd_per_cell)
eff = (RUN_CFG.effort_large if n_val > RUN_CFG.effort_col_switch
       else RUN_CFG.effort_small)

print(f"form    {_row['form_name']}  [{_row['schema_name']}]")
print(f'paper   {MD_PATH.name}  ({len(MD_PATH.read_text()):,} chars)')
print(f'field   {field_def["name"]}: {len(cols)} cols, {len(anchors)} anchors, {n_val} value cols')
print(f'anchors {sorted(anchors)}')
print(f'caps    session ${sess_cap:.2f} | extraction '
      f'${RUN_CFG.budget_usd_extraction_cap:.2f} | effort {eff}')
print()

if not REALLY_RUN:
    print('skipped (REALLY_RUN is False)')
else:
    _check_budget()
    res = await A.extract_table_from_markdown(
        markdown_content=MD_PATH.read_text(),
        sig_def=sig_def, field_def=field_def,
        paper_hint=MD_PATH.stem, cfg=RUN_CFG,
    )
    _charge(res.cost_usd)
    print(f'status     {res.status}')
    print(f'rows       emitted={res.n_rows} planned={res.plan_rows} '
          f'missing={res.rows_missing} withdrawn={res.rows_withdrawn}')
    print(f'repairs    {res.repair_rounds}   turns {res.num_turns}   subtype {res.subtype}')
    print(f'quotes     checked={res.quotes_checked} '
          f'failed_inloop={res.quotes_failed_inloop} '
          f'failed_grounding={res.cells_failed_grounding}')
    print(f'provenance table={res.cells_from_table} prose={res.cells_from_prose} '
          f'nr={res.cells_nr}')
    print(f'cost       ${res.cost_usd:.4f}  ({res.duration_ms/1000:.0f}s)')
    print(f'notes      {res.notes or "-"}')
    print(f'trace      {res.trace_path or "-"}')
    show_table(res, field_def)
    print()
    print('--- one cell in full (value + verbatim quote + location) ---')
    show_cell(res, field_def)


### 10. Read the trace  *(free — run after cell 9)*

Every step the agent took: its row plan, each tool call with arguments, each quote
check result, any repair round, and its reasoning if effort produced any.


In [ ]:
from pathlib import Path as _P
tdir = _P('/tmp/evistream_agentic_traces')
files = sorted(tdir.glob('*.jsonl'), key=lambda f: f.stat().st_mtime, reverse=True) if tdir.exists() else []
if not files:
    print('no traces yet — run cell 9 with REALLY_RUN=True and trace enabled')
else:
    latest = files[0]
    print('trace:', latest.name, f'({latest.stat().st_size:,} bytes)')
    print('=' * 78)
    for line in latest.read_text().splitlines():
        try:
            e = json.loads(line)
        except Exception:
            continue
        k = e.get('event') or e.get('type') or '?'
        if k in ('tool_use', 'CALL'):
            print(f"CALL   {e.get('tool') or e.get('name')}  "
                  f"{json.dumps(e.get('input') or e.get('args') or {})[:200]}")
        elif k in ('tool_result', 'RESULT'):
            print(f"RESULT {str(e.get('content') or e.get('result'))[:200]}")
        elif k in ('thinking',):
            print(f"THINK  {str(e.get('text'))[:220]}")
        elif k in ('text', 'SAY'):
            print(f"SAY    {str(e.get('text'))[:220]}")
        elif 'repair' in str(k):
            print(f"REPAIR {json.dumps(e)[:220]}")
        elif k == 'done':
            print('-' * 78)
            print('DONE  ', json.dumps(e)[:400])
        else:
            print(f"{k:6s} {json.dumps(e)[:180]}")
